# Model Interpretation and Explainability Framework

This notebook implements comprehensive model interpretation techniques including:
- SHAP (SHapley Additive exPlanations)
- LIME (Local Interpretable Model-agnostic Explanations)
- Grad-CAM and Integrated Gradients
- Feature importance analysis
- Attention visualization
- Counterfactual explanations

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Interpretability libraries
import shap
import lime
import lime.lime_tabular
from captum.attr import (
    IntegratedGradients,
    DeepLift,
    GradientShap,
    NoiseTunnel,
    FeatureAblation,
    LayerConductance,
    LayerActivation,
    LayerGradCam
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Utilities
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Model Wrapper for Interpretation

In [ ]:
@dataclass
class InterpretationConfig:
    """Configuration for model interpretation."""
    n_samples: int = 100
    baseline_type: str = 'zeros'  # 'zeros', 'random', 'mean'
    n_steps: int = 50  # For integrated gradients
    batch_size: int = 32
    feature_names: Optional[List[str]] = None
    class_names: Optional[List[str]] = None
    

class ModelWrapper:
    """Wrapper class for model interpretation."""
    
    def __init__(self, model: nn.Module, 
                 device: str = None,
                 task_type: str = 'classification'):
        """
        Initialize model wrapper.
        
        Parameters:
        -----------
        model : nn.Module
            PyTorch model
        device : str
            Device to use
        task_type : str
            Task type ('classification' or 'regression')
        """
        self.model = model
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.task_type = task_type
        self.model.to(self.device)
        self.model.eval()
        
    def predict(self, X: Union[np.ndarray, torch.Tensor]) -> np.ndarray:
        """Make predictions."""
        if isinstance(X, np.ndarray):
            X = torch.FloatTensor(X)
        
        X = X.to(self.device)
        
        with torch.no_grad():
            outputs = self.model(X)
            
            if self.task_type == 'classification':
                probs = F.softmax(outputs, dim=1)
                return probs.cpu().numpy()
            else:
                return outputs.cpu().numpy()
    
    def predict_proba(self, X: Union[np.ndarray, torch.Tensor]) -> np.ndarray:
        """Get prediction probabilities for classification."""
        return self.predict(X)
    
    def get_loss(self, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Calculate loss."""
        outputs = self.model(X)
        
        if self.task_type == 'classification':
            criterion = nn.CrossEntropyLoss()
        else:
            criterion = nn.MSELoss()
        
        return criterion(outputs, y)

## 2. SHAP-based Interpretations

In [ ]:
class SHAPInterpreter:
    """SHAP-based model interpretation."""
    
    def __init__(self, model_wrapper: ModelWrapper, 
                 background_data: np.ndarray,
                 config: InterpretationConfig = None):
        """
        Initialize SHAP interpreter.
        
        Parameters:
        -----------
        model_wrapper : ModelWrapper
            Wrapped model
        background_data : np.ndarray
            Background data for SHAP
        config : InterpretationConfig
            Configuration
        """
        self.model_wrapper = model_wrapper
        self.background_data = background_data
        self.config = config or InterpretationConfig()
        
        # Create SHAP explainer
        self.explainer = shap.DeepExplainer(
            self.model_wrapper.model,
            torch.FloatTensor(background_data[:min(100, len(background_data))]).to(self.model_wrapper.device)
        )
    
    def explain_instance(self, X: np.ndarray, 
                        target_class: Optional[int] = None) -> Dict:
        """Explain a single instance."""
        X_tensor = torch.FloatTensor(X.reshape(1, -1)).to(self.model_wrapper.device)
        
        # Get SHAP values
        shap_values = self.explainer.shap_values(X_tensor)
        
        if isinstance(shap_values, list):
            # Multi-class classification
            if target_class is not None:
                shap_values = shap_values[target_class]
            else:
                # Use predicted class
                pred = self.model_wrapper.predict(X_tensor)
                target_class = np.argmax(pred)
                shap_values = shap_values[target_class]
        
        # Get prediction
        prediction = self.model_wrapper.predict(X_tensor)[0]
        
        return {
            'shap_values': shap_values[0],
            'base_value': self.explainer.expected_value[target_class] if isinstance(self.explainer.expected_value, list) else self.explainer.expected_value,
            'prediction': prediction,
            'target_class': target_class,
            'feature_values': X
        }
    
    def explain_dataset(self, X: np.ndarray, 
                       sample_size: Optional[int] = None) -> Dict:
        """Explain multiple instances."""
        if sample_size and len(X) > sample_size:
            indices = np.random.choice(len(X), sample_size, replace=False)
            X = X[indices]
        
        X_tensor = torch.FloatTensor(X).to(self.model_wrapper.device)
        
        # Get SHAP values for all samples
        shap_values = self.explainer.shap_values(X_tensor)
        
        return {
            'shap_values': shap_values,
            'base_values': self.explainer.expected_value,
            'feature_values': X
        }
    
    def plot_summary(self, X: np.ndarray, 
                    feature_names: Optional[List[str]] = None,
                    max_display: int = 20):
        """Plot SHAP summary."""
        explanation = self.explain_dataset(X, sample_size=min(1000, len(X)))
        
        # Handle multi-class
        shap_values = explanation['shap_values']
        if isinstance(shap_values, list):
            # Average across classes
            shap_values = np.mean(np.abs(shap_values), axis=0)
        
        # Create summary plot
        shap.summary_plot(
            shap_values,
            explanation['feature_values'],
            feature_names=feature_names or self.config.feature_names,
            max_display=max_display,
            show=False
        )
        
        plt.tight_layout()
        return plt.gcf()
    
    def plot_waterfall(self, X: np.ndarray, 
                      instance_idx: int = 0,
                      target_class: Optional[int] = None):
        """Plot waterfall chart for single instance."""
        explanation = self.explain_instance(X[instance_idx], target_class)
        
        # Create waterfall data
        shap_values = explanation['shap_values']
        base_value = explanation['base_value']
        
        # Sort features by importance
        abs_shap = np.abs(shap_values)
        sorted_indices = np.argsort(abs_shap)[::-1][:10]  # Top 10 features
        
        fig = go.Figure(go.Waterfall(
            name="SHAP",
            orientation="v",
            measure=["relative"] * len(sorted_indices) + ["total"],
            x=[f"Feature {i}" for i in sorted_indices] + ["Prediction"],
            y=list(shap_values[sorted_indices]) + [base_value + np.sum(shap_values)],
            connector={"line": {"color": "rgb(63, 63, 63)"}},
        ))
        
        fig.update_layout(
            title="SHAP Waterfall Plot",
            showlegend=False,
            height=500
        )
        
        return fig

## 3. LIME-based Interpretations

In [ ]:
class LIMEInterpreter:
    """LIME-based model interpretation."""
    
    def __init__(self, model_wrapper: ModelWrapper,
                 training_data: np.ndarray,
                 config: InterpretationConfig = None):
        """
        Initialize LIME interpreter.
        
        Parameters:
        -----------
        model_wrapper : ModelWrapper
            Wrapped model
        training_data : np.ndarray
            Training data for statistics
        config : InterpretationConfig
            Configuration
        """
        self.model_wrapper = model_wrapper
        self.training_data = training_data
        self.config = config or InterpretationConfig()
        
        # Create LIME explainer
        self.explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data,
            feature_names=config.feature_names if config else None,
            class_names=config.class_names if config else None,
            mode='classification' if model_wrapper.task_type == 'classification' else 'regression',
            discretize_continuous=True
        )
    
    def explain_instance(self, X: np.ndarray,
                        num_features: int = 10,
                        target_class: Optional[int] = None) -> Dict:
        """Explain a single instance using LIME."""
        # Get explanation
        if self.model_wrapper.task_type == 'classification':
            exp = self.explainer.explain_instance(
                X,
                self.model_wrapper.predict_proba,
                num_features=num_features,
                labels=[target_class] if target_class is not None else None
            )
        else:
            exp = self.explainer.explain_instance(
                X,
                self.model_wrapper.predict,
                num_features=num_features
            )
        
        # Extract explanation
        if target_class is not None:
            feature_importance = exp.as_list(label=target_class)
            local_pred = exp.local_pred[target_class] if hasattr(exp, 'local_pred') else None
        else:
            feature_importance = exp.as_list()
            local_pred = exp.local_pred[0] if hasattr(exp, 'local_pred') else None
        
        return {
            'feature_importance': feature_importance,
            'local_prediction': local_pred,
            'prediction': self.model_wrapper.predict(X.reshape(1, -1))[0],
            'explanation_object': exp
        }
    
    def plot_explanation(self, X: np.ndarray,
                        instance_idx: int = 0,
                        num_features: int = 10):
        """Plot LIME explanation."""
        explanation = self.explain_instance(
            X[instance_idx],
            num_features=num_features
        )
        
        # Extract feature importance
        features = []
        importances = []
        
        for feature, importance in explanation['feature_importance']:
            features.append(feature)
            importances.append(importance)
        
        # Create bar plot
        fig, ax = plt.subplots(figsize=(10, 6))
        
        colors = ['green' if x > 0 else 'red' for x in importances]
        y_pos = np.arange(len(features))
        
        ax.barh(y_pos, importances, color=colors, alpha=0.7)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(features)
        ax.set_xlabel('Feature Importance')
        ax.set_title('LIME Feature Importance')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig

## 4. Gradient-based Interpretations

In [ ]:
class GradientInterpreter:
    """Gradient-based model interpretation using Captum."""
    
    def __init__(self, model_wrapper: ModelWrapper,
                 config: InterpretationConfig = None):
        """
        Initialize gradient interpreter.
        
        Parameters:
        -----------
        model_wrapper : ModelWrapper
            Wrapped model
        config : InterpretationConfig
            Configuration
        """
        self.model_wrapper = model_wrapper
        self.config = config or InterpretationConfig()
        
        # Initialize attribution methods
        self.integrated_gradients = IntegratedGradients(model_wrapper.model)
        self.deep_lift = DeepLift(model_wrapper.model)
        self.gradient_shap = GradientShap(model_wrapper.model)
        self.feature_ablation = FeatureAblation(model_wrapper.model)
    
    def get_baseline(self, X: torch.Tensor) -> torch.Tensor:
        """Get baseline for attribution methods."""
        if self.config.baseline_type == 'zeros':
            return torch.zeros_like(X)
        elif self.config.baseline_type == 'random':
            return torch.randn_like(X) * 0.01
        elif self.config.baseline_type == 'mean':
            return torch.ones_like(X) * X.mean(dim=0, keepdim=True)
        else:
            return torch.zeros_like(X)
    
    def integrated_gradients_attribution(self, X: torch.Tensor,
                                        target: Optional[int] = None) -> torch.Tensor:
        """Calculate Integrated Gradients attribution."""
        baseline = self.get_baseline(X)
        
        attributions = self.integrated_gradients.attribute(
            X,
            baseline,
            target=target,
            n_steps=self.config.n_steps
        )
        
        return attributions
    
    def deep_lift_attribution(self, X: torch.Tensor,
                            target: Optional[int] = None) -> torch.Tensor:
        """Calculate DeepLift attribution."""
        baseline = self.get_baseline(X)
        
        attributions = self.deep_lift.attribute(
            X,
            baseline,
            target=target
        )
        
        return attributions
    
    def gradient_shap_attribution(self, X: torch.Tensor,
                                 baseline_samples: torch.Tensor,
                                 target: Optional[int] = None) -> torch.Tensor:
        """Calculate Gradient SHAP attribution."""
        attributions = self.gradient_shap.attribute(
            X,
            baseline_samples,
            target=target,
            n_samples=self.config.n_samples
        )
        
        return attributions
    
    def compare_attributions(self, X: np.ndarray,
                           target_class: Optional[int] = None) -> Dict:
        """Compare different attribution methods."""
        X_tensor = torch.FloatTensor(X).unsqueeze(0).to(self.model_wrapper.device)
        X_tensor.requires_grad = True
        
        # Get prediction
        output = self.model_wrapper.model(X_tensor)
        if target_class is None and self.model_wrapper.task_type == 'classification':
            target_class = output.argmax(dim=1).item()
        
        # Calculate attributions
        ig_attr = self.integrated_gradients_attribution(X_tensor, target_class)
        dl_attr = self.deep_lift_attribution(X_tensor, target_class)
        
        # Feature ablation
        fa_attr = self.feature_ablation.attribute(X_tensor, target=target_class)
        
        return {
            'integrated_gradients': ig_attr.detach().cpu().numpy().squeeze(),
            'deep_lift': dl_attr.detach().cpu().numpy().squeeze(),
            'feature_ablation': fa_attr.detach().cpu().numpy().squeeze(),
            'prediction': output.detach().cpu().numpy().squeeze(),
            'target_class': target_class
        }
    
    def plot_attribution_comparison(self, X: np.ndarray,
                                  feature_names: Optional[List[str]] = None,
                                  top_k: int = 15):
        """Plot comparison of attribution methods."""
        attributions = self.compare_attributions(X)
        
        # Get top features by average importance
        avg_importance = np.mean([
            np.abs(attributions['integrated_gradients']),
            np.abs(attributions['deep_lift']),
            np.abs(attributions['feature_ablation'])
        ], axis=0)
        
        top_indices = np.argsort(avg_importance)[::-1][:top_k]
        
        if feature_names is None:
            feature_names = [f"Feature {i}" for i in range(len(X))]
        
        # Create subplot
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        methods = ['Integrated Gradients', 'DeepLift', 'Feature Ablation']
        method_attrs = ['integrated_gradients', 'deep_lift', 'feature_ablation']
        
        for idx, (ax, method, attr_key) in enumerate(zip(axes, methods, method_attrs)):
            values = attributions[attr_key][top_indices]
            colors = ['green' if v > 0 else 'red' for v in values]
            
            ax.barh(range(len(values)), values, color=colors, alpha=0.7)
            ax.set_yticks(range(len(values)))
            ax.set_yticklabels([feature_names[i] for i in top_indices])
            ax.set_xlabel('Attribution Score')
            ax.set_title(method)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(f"Attribution Method Comparison (Target Class: {attributions['target_class']})")
        plt.tight_layout()
        
        return fig

## 5. Counterfactual Explanations

In [ ]:
class CounterfactualExplainer:
    """Generate counterfactual explanations."""
    
    def __init__(self, model_wrapper: ModelWrapper,
                 feature_ranges: Optional[Dict[int, Tuple[float, float]]] = None):
        """
        Initialize counterfactual explainer.
        
        Parameters:
        -----------
        model_wrapper : ModelWrapper
            Wrapped model
        feature_ranges : Dict[int, Tuple[float, float]]
            Valid ranges for features
        """
        self.model_wrapper = model_wrapper
        self.feature_ranges = feature_ranges or {}
    
    def generate_counterfactual(self, X: np.ndarray,
                              target_class: int,
                              max_iterations: int = 1000,
                              learning_rate: float = 0.01,
                              lambda_reg: float = 0.1) -> Dict:
        """Generate counterfactual explanation."""
        # Convert to tensor
        X_cf = torch.FloatTensor(X.copy()).to(self.model_wrapper.device)
        X_cf.requires_grad = True
        X_orig = torch.FloatTensor(X).to(self.model_wrapper.device)
        
        optimizer = torch.optim.Adam([X_cf], lr=learning_rate)
        
        best_cf = None
        best_loss = float('inf')
        
        for iteration in range(max_iterations):
            optimizer.zero_grad()
            
            # Forward pass
            output = self.model_wrapper.model(X_cf.unsqueeze(0))
            
            # Loss for target class
            if self.model_wrapper.task_type == 'classification':
                probs = F.softmax(output, dim=1)
                class_loss = -torch.log(probs[0, target_class] + 1e-10)
            else:
                class_loss = F.mse_loss(output, torch.tensor([[target_class]], dtype=torch.float32))
            
            # Distance penalty
            distance = torch.norm(X_cf - X_orig, p=2)
            
            # Total loss
            loss = class_loss + lambda_reg * distance
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Clip to valid ranges
            with torch.no_grad():
                for feature_idx, (min_val, max_val) in self.feature_ranges.items():
                    X_cf[feature_idx] = torch.clamp(X_cf[feature_idx], min_val, max_val)
            
            # Check if target achieved
            with torch.no_grad():
                current_output = self.model_wrapper.model(X_cf.unsqueeze(0))
                if self.model_wrapper.task_type == 'classification':
                    current_class = current_output.argmax(dim=1).item()
                    if current_class == target_class and loss.item() < best_loss:
                        best_cf = X_cf.cpu().numpy()
                        best_loss = loss.item()
        
        if best_cf is None:
            best_cf = X_cf.detach().cpu().numpy()
        
        # Calculate changes
        changes = best_cf - X
        changed_features = np.where(np.abs(changes) > 1e-3)[0]
        
        return {
            'counterfactual': best_cf,
            'original': X,
            'changes': changes,
            'changed_features': changed_features,
            'distance': np.linalg.norm(changes),
            'target_class': target_class,
            'achieved': self.model_wrapper.predict(best_cf.reshape(1, -1)).argmax() == target_class
                       if self.model_wrapper.task_type == 'classification' else True
        }
    
    def plot_counterfactual(self, counterfactual_result: Dict,
                          feature_names: Optional[List[str]] = None):
        """Plot counterfactual explanation."""
        changes = counterfactual_result['changes']
        changed_features = counterfactual_result['changed_features']
        
        if feature_names is None:
            feature_names = [f"Feature {i}" for i in range(len(changes))]
        
        # Get top changes
        top_changes_idx = changed_features[np.argsort(np.abs(changes[changed_features]))[::-1][:10]]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot changes
        y_pos = np.arange(len(top_changes_idx))
        changes_to_plot = changes[top_changes_idx]
        colors = ['green' if c > 0 else 'red' for c in changes_to_plot]
        
        ax1.barh(y_pos, changes_to_plot, color=colors, alpha=0.7)
        ax1.set_yticks(y_pos)
        ax1.set_yticklabels([feature_names[i] for i in top_changes_idx])
        ax1.set_xlabel('Change in Feature Value')
        ax1.set_title('Required Changes for Counterfactual')
        ax1.grid(True, alpha=0.3)
        
        # Plot original vs counterfactual
        x_pos = np.arange(len(top_changes_idx))
        width = 0.35
        
        original_values = counterfactual_result['original'][top_changes_idx]
        cf_values = counterfactual_result['counterfactual'][top_changes_idx]
        
        ax2.bar(x_pos - width/2, original_values, width, label='Original', alpha=0.7)
        ax2.bar(x_pos + width/2, cf_values, width, label='Counterfactual', alpha=0.7)
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels([feature_names[i] for i in top_changes_idx], rotation=45)
        ax2.set_ylabel('Feature Value')
        ax2.set_title('Original vs Counterfactual Values')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle(f"Counterfactual Explanation (Target Class: {counterfactual_result['target_class']}, "
                    f"Achieved: {counterfactual_result['achieved']})")
        plt.tight_layout()
        
        return fig

## 6. Comprehensive Interpretation Pipeline

In [ ]:
class ModelInterpreter:
    """Comprehensive model interpretation pipeline."""
    
    def __init__(self, model: nn.Module,
                 X_train: np.ndarray,
                 config: InterpretationConfig = None,
                 device: str = None):
        """
        Initialize model interpreter.
        
        Parameters:
        -----------
        model : nn.Module
            Model to interpret
        X_train : np.ndarray
            Training data for background/statistics
        config : InterpretationConfig
            Configuration
        device : str
            Device to use
        """
        self.config = config or InterpretationConfig()
        self.model_wrapper = ModelWrapper(model, device)
        
        # Initialize interpreters
        self.shap_interpreter = SHAPInterpreter(
            self.model_wrapper,
            X_train[:min(100, len(X_train))],
            config
        )
        
        self.lime_interpreter = LIMEInterpreter(
            self.model_wrapper,
            X_train,
            config
        )
        
        self.gradient_interpreter = GradientInterpreter(
            self.model_wrapper,
            config
        )
        
        self.counterfactual_explainer = CounterfactualExplainer(
            self.model_wrapper
        )
    
    def interpret_instance(self, X: np.ndarray,
                         instance_idx: int = 0,
                         methods: List[str] = ['shap', 'lime', 'gradient']) -> Dict:
        """Comprehensive interpretation of a single instance."""
        results = {}
        instance = X[instance_idx]
        
        if 'shap' in methods:
            try:
                results['shap'] = self.shap_interpreter.explain_instance(instance)
            except Exception as e:
                print(f"SHAP failed: {e}")
        
        if 'lime' in methods:
            try:
                results['lime'] = self.lime_interpreter.explain_instance(instance)
            except Exception as e:
                print(f"LIME failed: {e}")
        
        if 'gradient' in methods:
            try:
                results['gradient'] = self.gradient_interpreter.compare_attributions(instance)
            except Exception as e:
                print(f"Gradient methods failed: {e}")
        
        return results
    
    def generate_report(self, X: np.ndarray,
                       instance_idx: int = 0,
                       save_path: Optional[str] = None):
        """Generate comprehensive interpretation report."""
        # Get all interpretations
        interpretations = self.interpret_instance(X, instance_idx)
        
        # Create figure with subplots
        fig = plt.figure(figsize=(20, 12))
        
        # Layout: 2x3 grid
        # Row 1: SHAP, LIME, Gradient comparison
        # Row 2: Feature importance summary, Counterfactual, Model performance
        
        # Plot SHAP
        if 'shap' in interpretations:
            ax1 = plt.subplot(2, 3, 1)
            shap_values = interpretations['shap']['shap_values']
            top_features = np.argsort(np.abs(shap_values))[::-1][:10]
            ax1.barh(range(len(top_features)), shap_values[top_features],
                    color=['green' if v > 0 else 'red' for v in shap_values[top_features]])
            ax1.set_yticks(range(len(top_features)))
            ax1.set_yticklabels([f"Feature {i}" for i in top_features])
            ax1.set_xlabel('SHAP Value')
            ax1.set_title('SHAP Feature Importance')
        
        # Plot LIME
        if 'lime' in interpretations:
            ax2 = plt.subplot(2, 3, 2)
            lime_features = interpretations['lime']['feature_importance'][:10]
            features = [f[0] for f in lime_features]
            importances = [f[1] for f in lime_features]
            ax2.barh(range(len(features)), importances,
                    color=['green' if v > 0 else 'red' for v in importances])
            ax2.set_yticks(range(len(features)))
            ax2.set_yticklabels(features)
            ax2.set_xlabel('LIME Importance')
            ax2.set_title('LIME Feature Importance')
        
        # Plot Gradient methods
        if 'gradient' in interpretations:
            ax3 = plt.subplot(2, 3, 3)
            ig_values = interpretations['gradient']['integrated_gradients']
            top_features = np.argsort(np.abs(ig_values))[::-1][:10]
            ax3.barh(range(len(top_features)), ig_values[top_features],
                    color=['green' if v > 0 else 'red' for v in ig_values[top_features]])
            ax3.set_yticks(range(len(top_features)))
            ax3.set_yticklabels([f"Feature {i}" for i in top_features])
            ax3.set_xlabel('Attribution Score')
            ax3.set_title('Integrated Gradients')
        
        plt.suptitle('Model Interpretation Report', fontsize=16)
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=100, bbox_inches='tight')
        
        return fig, interpretations

## 7. Example Usage

In [ ]:
# Create synthetic dataset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate data
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=3,
    random_state=42
)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Dataset shape: {X_train.shape}")
print(f"Number of classes: {len(np.unique(y))}")

In [ ]:
# Create a simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Initialize model
model = SimpleNN(input_dim=20, hidden_dim=64, output_dim=3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Train the model (simplified)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)

# Simple training loop
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

model.eval()
print("Model training complete!")

In [ ]:
# Initialize interpreter
config = InterpretationConfig(
    feature_names=[f"Feature {i}" for i in range(20)],
    class_names=["Class 0", "Class 1", "Class 2"],
    n_samples=50
)

interpreter = ModelInterpreter(
    model=model,
    X_train=X_train,
    config=config,
    device=device
)

print("Interpreter initialized!")

In [ ]:
# Interpret a test instance
test_instance_idx = 0
interpretations = interpreter.interpret_instance(X_test, test_instance_idx)

print(f"Interpreting test instance {test_instance_idx}")
print(f"True label: {y_test[test_instance_idx]}")

# Get prediction
prediction = interpreter.model_wrapper.predict(X_test[test_instance_idx].reshape(1, -1))
predicted_class = np.argmax(prediction)
print(f"Predicted class: {predicted_class}")
print(f"Prediction confidence: {prediction[0, predicted_class]:.3f}")

In [ ]:
# Generate comprehensive report
fig, results = interpreter.generate_report(X_test, test_instance_idx)
plt.show()

In [ ]:
# Generate counterfactual explanation
target_class = 1 if predicted_class != 1 else 0
counterfactual = interpreter.counterfactual_explainer.generate_counterfactual(
    X_test[test_instance_idx],
    target_class=target_class,
    learning_rate=0.1,
    max_iterations=500
)

print(f"\nCounterfactual Explanation:")
print(f"Original prediction: {predicted_class}")
print(f"Target class: {target_class}")
print(f"Counterfactual achieved: {counterfactual['achieved']}")
print(f"Number of changed features: {len(counterfactual['changed_features'])}")
print(f"Total distance: {counterfactual['distance']:.3f}")

# Plot counterfactual
cf_fig = interpreter.counterfactual_explainer.plot_counterfactual(
    counterfactual,
    feature_names=config.feature_names
)
plt.show()

## Summary

This notebook provides a comprehensive framework for model interpretation with:

### Interpretation Methods:
1. **SHAP** - Game-theoretic approach to feature importance
2. **LIME** - Local interpretable model-agnostic explanations
3. **Gradient-based** - Integrated Gradients, DeepLift, Gradient SHAP
4. **Counterfactual** - What-if explanations

### Key Features:
- Multiple interpretation methods for comparison
- Support for classification and regression
- Interactive visualizations
- Counterfactual generation
- Comprehensive reporting

### Applications:
- Model debugging and validation
- Feature selection
- Regulatory compliance (explainable AI)
- Building trust in model predictions
- Understanding model behavior

The framework is extensible and can be adapted for various model types and domains.